In [1]:
"""" This Cell is used for Import modules and Creating connection for mysql 
database and manageing path of pdf"""

import tkinter as tk
from tkinter import messagebox, simpledialog, ttk
from reportlab.lib.pagesizes import A4
from reportlab.platypus import Table, TableStyle, SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
import mysql.connector
from mysql.connector import Error
import hashlib
import os
import random
import re
import threading
import time

MYSQL_CONFIG = {
    'host': 'localhost',
    'user': 'root',    
    'password': '',     
    'database': 'electricity_system'
}

PDF_SAVE_PATH = "/home/kali/Downloads/"  

In [2]:
# This Cell is Used for handling databse and user management

class Database:
    def __init__(self):
        self.conn = None
        self.connect()

    def connect(self):
        try:
            self.conn = mysql.connector.connect(**MYSQL_CONFIG)
        except Error as e:
            messagebox.showerror("Database Error", f"Error connecting to MySQL: {e}")
            exit()

    def close(self):
        if self.conn.is_connected():
            self.conn.close()

    def generate_unique_customer_id(self):
        while True:
            cust_id = f"{random.randint(10000, 99999)}"
            cursor = self.conn.cursor()
            cursor.execute("SELECT id FROM users WHERE customer_id=%s", (cust_id,))
            exists = cursor.fetchone()
            cursor.close()
            if not exists:
                return cust_id

    def create_user(self, email, password):
        try:
            cursor = self.conn.cursor()
            hashed_pw = self.hash_password(password)
            customer_id = self.generate_unique_customer_id()
            if not self.is_valid_email(email):
                return False, None
            cursor.execute(
                "INSERT INTO users (email, password, customer_id) VALUES (%s, %s, %s)", 
                (email, hashed_pw, customer_id)
            )
            self.conn.commit()
            cursor.close()
            return True, customer_id
        except Error as e:
            if e.errno == 1062:
                return False, None
            messagebox.showerror("DB Error", f"Error creating user: {e}")
            return False, None

    def verify_user(self, email, password):
        try:
            cursor = self.conn.cursor()
            cursor.execute("SELECT id, password, customer_id FROM users WHERE email=%s", (email,))
            row = cursor.fetchone()
            cursor.close()
            if row:
                stored_hash = row[1]
                input_hash = self.hash_password(password)
                if stored_hash == input_hash:
                    user_id = row[0]
                    customer_id = row[2]
                    return True, user_id, customer_id
            return False, None, None
        except Error as e:
            messagebox.showerror("DB Error", f"Error verifying user: {e}")
            return False, None, None

    def add_transaction(self, user_id, units, amount):
        try:
            cursor = self.conn.cursor()
            cursor.execute(
                "INSERT INTO transactions (user_id, units_consumed, amount) VALUES (%s, %s, %s)", 
                (user_id, units, amount)
            )
            self.conn.commit()
            cursor.close()
        except Error as e:
            messagebox.showerror("DB Error", f"Error adding transaction: {e}")

    def get_transactions(self, user_id):
        try:
            cursor = self.conn.cursor()
            cursor.execute(
                "SELECT units_consumed, amount, timestamp FROM transactions WHERE user_id=%s ORDER BY timestamp DESC", 
                (user_id,)
            )
            rows = cursor.fetchall()
            cursor.close()
            return rows
        except Error as e:
            messagebox.showerror("DB Error", f"Error fetching transactions: {e}")
            return []

    def get_user_by_customer_id(self, customer_id):
        try:
            cursor = self.conn.cursor()
            cursor.execute("SELECT id, email FROM users WHERE customer_id=%s", (customer_id,))
            user = cursor.fetchone()
            cursor.close()
            return user
        except Error as e:
            messagebox.showerror("DB Error", f"Error fetching user by customer ID: {e}")
            return None

    @staticmethod
    def hash_password(password):
        return hashlib.sha256(password.encode()).hexdigest()

    @staticmethod
    def is_valid_email(email):
        pattern = r'^[\w\.-]+@[\w\.-]+\.\w+$'
        return re.match(pattern, email) is not None

In [3]:
# This Cell is Used for Manging the Main Application class with session

class App(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Electricity Billing System")
        self.geometry("900x600")
        self.resizable(False, False)
        self.db = Database()

        self.current_user_id = None
        self.current_customer_id = None
        self.session_start_time = None
        self.session_timeout = 600  # 10 minutes
        self.check_session_thread = None
        self.stop_thread = False

        self.frames = {}
        for F in (LoginFrame, SignupFrame, DashboardFrame):
            frame = F(self)
            self.frames[F] = frame
            frame.pack_forget()

        self.show_frame(LoginFrame)
        self.start_session_monitor()

    def show_frame(self, frame_class):
        for frame in self.frames.values():
            frame.pack_forget()
        self.frames[frame_class].pack(fill="both", expand=True)
        if frame_class == SignupFrame:
            self.frames[SignupFrame].reset_signup()

    def set_session(self, user_id, customer_id):
        self.current_user_id = user_id
        self.current_customer_id = customer_id
        self.session_start_time = time.time()

    def clear_session(self):
        self.current_user_id = None
        self.current_customer_id = None
        self.session_start_time = None

    def start_session_monitor(self):
        def monitor():
            while not self.stop_thread:
                if self.current_user_id is not None:
                    elapsed = time.time() - self.session_start_time
                    if elapsed > self.session_timeout:
                        self.after(0, self.expire_session)
                time.sleep(1)
        self.check_session_thread = threading.Thread(target=monitor, daemon=True)
        self.check_session_thread.start()

    def expire_session(self):
        messagebox.showinfo("Session Expired", "Your session has expired Please Relogning.")
        self.clear_session()
        self.show_frame(LoginFrame)

    def on_closing(self):
        self.stop_thread = True
        self.db.close()
        self.destroy()

In [4]:
# This Cell is Used for Creating The Sinup Frame

class SignupFrame(tk.Frame):
    def __init__(self, master):
        super().__init__(master)
        tk.Label(self, text="Sign Up", font=("Arial", 20, "bold")).pack(pady=20)
        tk.Label(self, text="Email:").pack()
        self.email_entry = tk.Entry(self, width=40)
        self.email_entry.pack(pady=5)
        tk.Label(self, text="Password:").pack()
        self.pass_entry = tk.Entry(self, width=40, show="*")
        self.pass_entry.pack(pady=5)
        tk.Label(self, text="Confirm Password:").pack()
        self.conf_pass_entry = tk.Entry(self, width=40, show="*")
        self.conf_pass_entry.pack(pady=5)
        tk.Button(self, text="Sign Up", command=self.signup).pack(pady=20)
        tk.Button(self, text="Back to Login", command=lambda: self.master.show_frame(LoginFrame)).pack()

    def reset_signup(self):
        self.email_entry.delete(0, tk.END)
        self.pass_entry.delete(0, tk.END)
        self.conf_pass_entry.delete(0, tk.END)

    def signup(self):
        email = self.email_entry.get().strip()
        password = self.pass_entry.get()
        confirm_password = self.conf_pass_entry.get()
        if not email or not password:
            messagebox.showerror("Error", "All fields are required.")
            return
        if not self.master.db.is_valid_email(email):
            messagebox.showerror("Error", "Please enter a valid email address.")
            return
        if password != confirm_password:
            messagebox.showerror("Error", "Passwords do not match.")
            return
        success, cust_id = self.master.db.create_user(email, password)
        if success:
            messagebox.showinfo("Success", "Account created! Please login.")
            self.master.show_frame(LoginFrame)
        else:
            messagebox.showerror("Error", "Email already exists or invalid.")
        self.reset_signup()

In [5]:
# This Cell is Used for Creating The Login Frame

class LoginFrame(tk.Frame):
    def __init__(self, master):
        super().__init__(master)
        tk.Label(self, text="Login", font=("Arial", 20, "bold")).pack(pady=20)
        tk.Label(self, text="Email:").pack()
        self.email_entry = tk.Entry(self, width=40)
        self.email_entry.pack(pady=5)
        tk.Label(self, text="Password:").pack()
        self.pass_entry = tk.Entry(self, width=40, show="*")
        self.pass_entry.pack(pady=5)
        tk.Button(self, text="Login", command=self.login).pack(pady=20)
        tk.Button(self, text="Go to Sign Up", command=lambda: self.master.show_frame(SignupFrame)).pack()

    def login(self):
        email = self.email_entry.get().strip()
        password = self.pass_entry.get()
        valid, user_id, customer_id = self.master.db.verify_user(email, password)
        if valid:
            self.master.set_session(user_id, customer_id)
            self.master.frames[DashboardFrame].load_user(user_id, email, customer_id)
            self.master.show_frame(DashboardFrame)
            self.email_entry.delete(0, tk.END)
            self.pass_entry.delete(0, tk.END)
        else:
            messagebox.showerror("Error", "Invalid email or password.")

In [6]:
# This Cell is Used For Creating Main Freame and Main Calculation

class DashboardFrame(tk.Frame):
    def __init__(self, master):
        super().__init__(master)
        self.user_id = None
        self.email = None
        self.customer_id = None
        self.bill_data = {}
        self.account_info = {}

        # navbar with welcome and logout text & btn
        top_frame = tk.Frame(self, bg="#2C3E50")
        top_frame.pack(fill="x", pady=5)
        self.welcome_label = tk.Label(top_frame, text="", font=("Arial", 16),fg="white", bg="#2C3E50")
        self.welcome_label.pack(side="left", padx=10, pady=8)
        tk.Button(top_frame, text="Logout", command=self.logout).pack(side="right", padx=10, pady=8)

        # Main container frame point
        container = tk.Frame(self)
        container.pack(fill="both", expand=True, padx=10, pady=5)

        # Left: Calculate Bill
        left_frame = tk.LabelFrame(container, text="Calculate Your Bill", 
                                   font=("Arial", 14, "bold"))
        left_frame.pack(side="left", fill="both", expand=True, padx=(0,5), pady=5)
        tk.Label(left_frame, text="Enter Units Consumed:", font=("Arial", 12)).pack(pady=10)
        self.units_entry = tk.Entry(left_frame, font=("Arial", 12))
        self.units_entry.pack(pady=5, padx=10)
        tk.Button(left_frame, text="Calculate", font=("Arial", 12, "bold"),
                  bg="#27AE60", fg="white", command=self.calculate_only).pack(pady=15)
        tk.Label(left_frame, text="Bill Summary:", font=("Arial", 12, "underline")).pack(pady=5)
        self.bill_text = tk.Text(left_frame, height=6, width=40, font=("Courier New", 11), bg="#ECF0F1", state="disabled")
        self.bill_text.pack(pady=5, padx=10)
        tk.Button(left_frame, text="Save as Bill PDF", font=("Arial", 12), command=self.save_as_pdf).pack(pady=5)

        # Right: Search and Transactions
        right_frame = tk.LabelFrame(container, text="Search Customer Id & View older Records", font=("Arial", 14, "bold"))
        right_frame.pack(side="right", fill="both", expand=True, padx=(5,0), pady=5)
        search_frame = tk.Frame(right_frame)
        search_frame.pack(pady=10, padx=5, fill="x")
        tk.Label(search_frame, text="Enter 5-digit Customer ID:", font=("Arial", 12)).pack(side="left")
        self.search_entry = tk.Entry(search_frame, width=10, font=("Arial", 12))
        self.search_entry.pack(side="left", padx=5)
        tk.Button(search_frame, text="Search", command=self.search_customer).pack(side="left")
        self.tree = ttk.Treeview(right_frame, columns=("units", "amount", "date"), show="headings")
        self.tree.heading("units", text="Units Consumed")
        self.tree.heading("amount", text="Amount (Rs.)")
        self.tree.heading("date", text="Date & Time")
        self.tree.column("units", anchor="center", width=100)
        self.tree.column("amount", anchor="center", width=100)
        self.tree.column("date", anchor="center", width=180)
        self.tree.pack(fill="both", expand=True, padx=10, pady=10)
        self.account_info_label = tk.Label(right_frame, text="", font=("Arial", 12), justify="left")
        self.account_info_label.pack(padx=10, pady=5, anchor="w")
        tk.Button(right_frame, text="Generate Complete Bill PDF", command=self.generate_account_pdf).pack(pady=5)

        self.master = master

    def load_user(self, user_id, email, customer_id):
        self.user_id = user_id
        self.email = email
        self.customer_id = customer_id
        self.welcome_label.config(text=f"Logged in as: {email} (Customer ID: {customer_id})")
        self.reset_display()

    def reset_display(self):
        self.bill_text.configure(state="normal")
        self.bill_text.delete(1.0, tk.END)
        self.bill_text.configure(state="disabled")
        self.units_entry.delete(0, tk.END)
        self.search_entry.delete(0, tk.END)
        self.account_info_label.config(text="")
        self.clear_table()
        self.bill_data = {}
        self.account_info = {}

    def logout(self):
        self.master.clear_session()
        self.master.show_frame(LoginFrame)

    def clear_table(self):
        for row in self.tree.get_children():
            self.tree.delete(row)

    def search_customer(self):
        customer_id = self.search_entry.get().strip()
        if len(customer_id) != 5 or not customer_id.isdigit():
            messagebox.showerror("Error", "Please enter a valid 5-digit Customer ID.")
            return
        user = self.master.db.get_user_by_customer_id(customer_id)
        if user:
            user_id, email = user
            transactions = self.master.db.get_transactions(user_id)
            self.display_transactions(transactions, customer_id, email)
        else:
            messagebox.showinfo("Not Found", "Customer ID not found.")
            self.clear_table()
            self.account_info_label.config(text="")

    def display_transactions(self, transactions, customer_id, email):
        self.clear_table()
        total_units = 0
        total_amount = 0.0
        for t in transactions:
            units, amount, timestamp = t
            self.tree.insert("", "end", values=(units, f"{amount:.2f}", str(timestamp)))
            total_units += units
            total_amount += amount

        self.account_info = {
            "id": customer_id,
            "email": email,
            "total_units": total_units,
            "total_amount": total_amount,
            "transactions": transactions
        }
        self.account_info_label.config(text=(
            f"Customer ID: {customer_id}\n"
            f"Email: {email}\n"
            f"Total Units Used: {total_units} units\n"
            f"Total Paid: Rs {total_amount:.2f}"
        ))

    def calculate_only(self):
        units_str = self.units_entry.get().strip()
        try:
            units = float(units_str)
            if units < 0:
                raise ValueError
        except:
            messagebox.showerror("Error", "Enter a valid non-negative number for units.")
            return
        amount = self.compute_bill(units)
        self.bill_text.configure(state="normal")
        self.bill_text.delete(1.0, tk.END)
        self.bill_text.insert(tk.END, f"Units Consumed: {units}\nTotal Bill: Rs {amount:.2f}")
        self.bill_text.configure(state="disabled")
        self.bill_data = {"user": self.email, "units": units, "amount": amount}

    def compute_bill(self, units):
        if units <= 50:
            return units * 3.15
        elif units <= 100:
            return 50 * 3.15 + (units - 50) * 3.60
        elif units <= 250:
            return 50 * 3.15 + 50 * 3.60 + (units - 100) * 4.25
        else:
            return 50 * 3.15 + 50 * 3.60 + 150 * 4.25 + (units - 250) * 5.20

    def save_as_pdf(self):
        if not self.bill_data:
            messagebox.showerror("Error", "Please calculate the bill first.")
            return
        filename = simpledialog.askstring("Save as", "Enter filename for PDF (without extension):")
        if filename:
            filepath = os.path.join(PDF_SAVE_PATH, f"{filename}.pdf")
            try:
                # Save transaction
                self.master.db.add_transaction(
                    self.master.current_user_id,
                    self.bill_data['units'],
                    self.bill_data['amount']
                )
                self.create_pdf(filepath)
                messagebox.showinfo("Success", f"Bill saved as {filepath}")
            except Exception as e:
                messagebox.showerror("Error", f"Failed to save PDF: {e}")

    def create_pdf(self, filepath):
        from reportlab.platypus import Table, TableStyle, SimpleDocTemplate, Paragraph, Spacer
        from reportlab.lib.styles import getSampleStyleSheet
        from reportlab.lib import colors
        doc = SimpleDocTemplate(filepath, pagesize=A4)
        styles = getSampleStyleSheet()
        elements = []

        elements.append(Paragraph("Electricity Bill", styles['Heading1']))
        elements.append(Spacer(1, 12))
        elements.append(Paragraph(f"User: {self.bill_data['user']}", styles['Normal']))
        elements.append(Spacer(1, 12))
        data = [
            ['Description', 'Value'],
            ['Units Consumed', f"{self.bill_data['units']}"],
            ['Total Bill (Rs.)', f"{self.bill_data['amount']:.2f}"]
        ]
        table = Table(data, colWidths=[200, 100])
        table.setStyle(TableStyle([
            ('BACKGROUND', (0,0), (-1,0), colors.grey),
            ('TEXTCOLOR',(0,0),(-1,0),colors.whitesmoke),
            ('ALIGN',(0,0),(-1,-1),'CENTER'),
            ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
            ('FONTSIZE', (0,0), (-1,0), 14),
            ('BOTTOMPADDING', (0,0), (-1,0), 12),
            ('BACKGROUND',(0,1),(-1,-1),colors.beige),
            ('GRID', (0,0), (-1,-1), 1, colors.black),
        ]))
        elements.append(table)
        doc.build(elements)

    def generate_account_pdf(self):
        if not self.account_info:
            messagebox.showerror("Error", "No account info to generate PDF. Search first.")
            return
        filename = simpledialog.askstring("Save as", "Enter filename for account PDF (without extension):")
        if filename:
            filepath = os.path.join(PDF_SAVE_PATH, f"{filename}.pdf")
            try:
                self.create_account_pdf(filepath)
                messagebox.showinfo("Success", f"Account PDF saved as {filepath}")
            except Exception as e:
                messagebox.showerror("Error", f"Failed to save PDF: {e}")

    def create_account_pdf(self, filepath):
        from reportlab.platypus import Table, TableStyle, SimpleDocTemplate, Paragraph, Spacer
        from reportlab.lib.styles import getSampleStyleSheet
        from reportlab.lib import colors
        doc = SimpleDocTemplate(filepath, pagesize=A4)
        styles = getSampleStyleSheet()
        elements = []

        elements.append(Paragraph("Customer Account Details", styles['Heading1']))
        elements.append(Spacer(1, 12))
        details_str = (
            f"<b>Customer ID:</b> {self.account_info['id']}<br/>"
            f"<b>Email:</b> {self.account_info['email']}<br/>"
            f"<b>Total Units Used:</b> {self.account_info['total_units']}<br/>"
            f"<b>Total Paid (Rs.):</b> {self.account_info['total_amount']:.2f}"
        )
        elements.append(Paragraph(details_str, styles['Normal']))
        elements.append(Spacer(1, 12))
        data = [['Units Consumed', 'Amount (Rs.)', 'Date & Time']]
        for t in self.account_info['transactions']:
            data.append([str(t[0]), f"{t[1]:.2f}", str(t[2])])
        table = Table(data, colWidths=[150, 150, 200])
        table.setStyle(TableStyle([
            ('BACKGROUND', (0,0), (-1,0), colors.grey),
            ('TEXTCOLOR',(0,0),(-1,0),colors.whitesmoke),
            ('ALIGN',(0,0),(-1,-1),'CENTER'),
            ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
            ('FONTSIZE', (0,0), (-1,0), 12),
            ('BOTTOMPADDING', (0,0), (-1,0), 10),
            ('BACKGROUND',(0,1),(-1,-1),colors.beige),
            ('GRID', (0,0), (-1,-1), 1, colors.black),
        ]))
        elements.append(table)
        elements.append(Spacer(1, 30))
        thank_you_style = ParagraphStyle(name='ThankYou', parent=styles['Heading1'])
        thank_you_style.alignment = 1
        thank_you_style.fontSize = 24
        thank_you_style.textColor = colors.black
        elements.append(Paragraph("Thank You", thank_you_style))
        doc.build(elements)

In [ ]:
# This Cell is Used for calling Main app and Exucting other Cells

if __name__ == "__main__":
    app = App()
    app.mainloop()

Exception in Tkinter callback
Traceback (most recent call last):
  File "/usr/lib/python3.13/tkinter/__init__.py", line 2077, in __call__
    return self.func(*args)
           ~~~~~~~~~^^^^^^^
  File "/tmp/ipykernel_5537/2548546161.py", line 37, in signup
    success, cust_id = self.master.db.create_user(email, password)
                       ~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_5537/121007228.py", line 31, in create_user
    cursor = self.conn.cursor()
             ^^^^^^^^^^^^^^^^
AttributeError: 'NoneType' object has no attribute 'cursor'
Exception in Tkinter callback
Traceback (most recent call last):
  File "/usr/lib/python3.13/tkinter/__init__.py", line 2077, in __call__
    return self.func(*args)
           ~~~~~~~~~^^^^^^^
  File "/tmp/ipykernel_5537/2548546161.py", line 37, in signup
    success, cust_id = self.master.db.create_user(email, password)
                       ~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_5537/1